# Qwen3-Embedding-0.6B — DIMER text embedding tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/qwen3-embedding-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/qwen3-embedding-pipeline/blob/main/tutorials/qwen3_embedding_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Qwen%2FQwen3--Embedding--0.6B-ffcc4d?style=flat)](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B)
[![Upstream](https://img.shields.io/badge/Upstream-QwenLM%2FQwen3--Embedding-181717?style=flat&logo=github&logoColor=white)](https://github.com/QwenLM/Qwen3-Embedding)
[![arXiv](https://img.shields.io/badge/arXiv-2506.05176-b31b1b.svg)](https://arxiv.org/abs/2506.05176)

**Profile:** `TASK-INFERENCE`
**Notebook specification:** DIMER Notebook Specification 1.0
**Capability:** text embeddings (1024-d, last-token pooled, L2-normalised, instruction-aware queries) using the pinned `Qwen/Qwen3-Embedding-0.6B` weights

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API rather than reimplementing model inference. At inference each text is tokenised with left padding and truncated at 8,192 tokens, a 0.6 B-parameter Qwen3 decoder encodes it, the hidden state of the **last token** is taken as the text's vector, and the pipeline L2-normalises it to unit length. Queries are prefixed with a task instruction (`Instruct: …\nQuery:`) because the model is instruction-aware; documents are embedded as-is. **Embeddings are representations, not predictions:** nothing is classified, ranked or decided, and there is no label space. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights and tokenizer, and this repository adds packaging, snapshot verification, input validation with named ceilings, the query/document formatting contract, a fixed output contract and the `cosine_similarity` helper. The default sample is the four sentences from the pinned upstream README; the cosine values shown for them are a qualitative check, not a benchmark claim.

**Learning objectives:** bootstrap the repository in a fresh runtime, resolve the immutable upstream model revision, prepare a small identified set of queries and documents and validate it against the pipeline's ceilings, embed queries and documents through the public API with the instruction contract, read the vectors correctly (shape, pooling, normalisation, per-text unit), compare a query with two documents by cosine as a qualitative check, exercise an optional BYOD path, and export identifiers alongside vectors plus provenance.

**This notebook does not demonstrate:** reranking (see the sibling Qwen3 reranker pipeline), text generation or chat, classification, clustering quality, retrieval evaluation (nDCG/recall need a labelled query–document set), Matryoshka dimension truncation (fixed at 1024 here), or any training.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available. **Precision differs by device:** the pipeline runs float32 on CPU and bfloat16 on CUDA, so cosine values can differ in the second or third decimal between the two. CPU is slow for large corpora but fine for a handful of sentences: the repository's model card records 6.6 s to load and 0.44 s to embed the four default texts on CPU in the Windows venv (Intel Core Ultra 9 275HX). The pinned `torch==2.14.0` install and the ~1.19 GB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python and NumPy; what a dense vector, a unit norm and cosine similarity are.
- **Data:** the default sample is four short English sentences (two queries, two documents) taken verbatim from the pinned upstream README, written into the notebook as string literals, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one UTF-8 text file with one document per non-empty line (at most 64 lines, each under 100,000 characters; text beyond 8,192 tokens is truncated and flagged) plus a query typed into the form. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** GitHub (repository clone) and the Hugging Face Hub (the package's `stage_missing_files` fetches the pinned checkpoint once, because the Git repository does not vendor the weights). No credentials are required.

## 1. Bootstrap the repository and pinned runtime

When the notebook is opened without a repository checkout, this cell clones the repository. Released notebooks default to `main`; automated candidate validation can set `DIMER_TUTORIAL_REF` to an immutable commit or review branch. The repository is installed as a regular (non-editable) package so it is importable in this same runtime; an editable install would only become importable after a restart. Model-facing dependencies (`torch`, `transformers`, `huggingface-hub`, `safetensors`, `numpy`) are pinned exactly in `pyproject.toml`. If installation replaces any package that this runtime has already imported, the cell fails with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the repository revision, Python, `torch` and `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/kurtvalcorza/qwen3-embedding-pipeline.git'
REPO_NAME = 'qwen3-embedding-pipeline'
REPO_REF = os.environ.get('DIMER_TUTORIAL_REF', 'main')
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '-q', REPO_URL, str(checkout)], check=True)
    if REPO_REF != 'main':
        subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', 'main'], check=True)
        subprocess.run(['git', '-C', str(checkout), 'pull', '--ff-only', '-q', 'origin', 'main'], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    # Every distribution that is already imported in this runtime is captured before installation,
    # whatever its name (PIL -> pillow), so a pinned install that replaces any loaded package is
    # detected. Distribution metadata is compared with metadata afterwards: torch.__version__ carries
    # a local build label (for example 2.14.0+cu130) that the distribution version omits.
    def _installed_version(distribution):
        try:
            return importlib.metadata.version(distribution)
        except importlib.metadata.PackageNotFoundError:
            return None
    _module_dists = importlib.metadata.packages_distributions()
    _loaded_dists = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded_dists}
    # Non-editable install: an editable (.pth) install is not importable until the
    # interpreter restarts, which a fresh hosted runtime cannot do mid-notebook.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(ROOT)], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

REPO_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import platform, torch, transformers
print({'repository': str(ROOT), 'repository_revision': REPO_SHA, 'requested_ref': REPO_REF, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Prepare the sample texts or optional BYOD

The default sample is **public and bundled in code**: the two queries and two documents from the pinned upstream README (Apache-2.0), each given a stable identifier (`q1`, `q2`, `d1`, `d2`) so every vector and similarity can be mapped back to its text. They exist to show the contract, not to measure anything; the repository's smoke run embedded exactly these and reproduced the README's printed cosine matrix to four decimals. BYOD is optional and disabled by default; when enabled, upload one UTF-8 text file (one document per line) and set `BYOD_QUERY`; documents are identified `d1…dN` in file order.

Before anything expensive runs, this cell surfaces the pipeline's operational ceilings — `MAX_BATCH` (64 texts per `embed` call), `MAX_TEXT_TOKENS` (8,192; longer texts are truncated and the result flags them), `MAX_TEXT_CHARS` (100,000; longer strings are rejected before tokenisation), `EMBEDDING_DIM` (1024) — and validates the sample against the batch, empty-text and character ceilings with clear messages; token-level truncation can only be reported after tokenisation, so the result's `truncated` flags are checked in Section 4. The query instruction (`DEFAULT_QUERY_INSTRUCTION`) is printed because it is part of the query vector: a different instruction produces a different embedding. Look for a dictionary naming the sample kind, the identifiers, character counts and the instruction.

In [ ]:
import hashlib
import io

import numpy as np

from qwen3_embedding_pipeline import DEFAULT_QUERY_INSTRUCTION, EMBEDDING_DIM, MAX_BATCH, MAX_TEXT_CHARS, MAX_TEXT_TOKENS

USE_BYOD = False  # @param {type:"boolean"}
BYOD_QUERY = 'What is the capital of China?'  # @param {type:"string"}

print({'ceilings': {'MAX_BATCH': MAX_BATCH, 'MAX_TEXT_TOKENS': MAX_TEXT_TOKENS, 'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'EMBEDDING_DIM': EMBEDDING_DIM}})
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    corpus_name = next(iter(uploaded))
    lines = [line.strip() for line in io.TextIOWrapper(io.BytesIO(uploaded[corpus_name]), encoding='utf-8')]
    documents = [line for line in lines if line]
    queries = [BYOD_QUERY.strip()]
    sample_kind = 'BYOD'
else:
    # Public sample: the pinned upstream README's example queries and documents, as string literals.
    queries = ['What is the capital of China?', 'Explain gravity']
    documents = [
        'The capital of China is Beijing.',
        'Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun.',
    ]
    corpus_name = 'upstream_readme_example'
    sample_kind = 'public (pinned upstream README example, bundled as literals)'

query_ids = [f'q{i + 1}' for i in range(len(queries))]
document_ids = [f'd{i + 1}' for i in range(len(documents))]
for label, texts in (('queries', queries), ('documents', documents)):
    if not 1 <= len(texts) <= MAX_BATCH:
        raise ValueError(f'{label}: {len(texts)} texts, but one embed() call accepts 1..{MAX_BATCH}; split the corpus and rerun this cell.')
    for text_id, text in zip(query_ids if label == 'queries' else document_ids, texts, strict=True):
        if not text.strip():
            raise ValueError(f'{text_id} is empty; remove blank entries and rerun this cell.')
        if len(text) > MAX_TEXT_CHARS:
            raise ValueError(f'{text_id} has {len(text)} characters, above MAX_TEXT_CHARS={MAX_TEXT_CHARS}; shorten or split it and rerun this cell.')
corpus_sha256 = hashlib.sha256('\n'.join(queries + documents).encode('utf-8')).hexdigest()
print({'sample_kind': sample_kind, 'name': corpus_name, 'query_ids': query_ids, 'document_ids': document_ids, 'chars': {i: len(t) for i, t in zip(query_ids + document_ids, queries + documents, strict=True)}, 'corpus_sha256': corpus_sha256, 'query_instruction': DEFAULT_QUERY_INSTRUCTION})

## 3. Stage, verify and resolve the pinned model

Model acquisition goes through the package, not the notebook. The public API pins the exact upstream revision (`MODEL_ID`/`MODEL_REVISION` are imported from the package, never typed here). The Git repository carries `weights/qwen3-embedding-0.6b/dimer-base-manifest.json`, `config.json`, the tokenizer files and the Sentence-Transformers pooling configs (10 small files) but git-ignores the 1.19 GB `model.safetensors`, so in a fresh clone `stage_missing_files(WEIGHTS_DIR, allow_download=True)` fetches exactly the manifest entries that are absent, at the pinned revision, into the snapshot directory — it prints the list it fetched (`[]` on a warm runtime) and refuses a manifest whose identity differs from the package pins. `verify_snapshot(WEIGHTS_DIR)` then re-hashes every manifest entry (size and SHA-256) and raises on the first mismatch; its returned dict is printed. Only then does `from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files with `local_files_only=True`, `trust_remote_code=False`, left padding, float32 on CPU / bfloat16 on CUDA — there is no fallback to a different download. The effective model identity, the embedding dimension and the device chosen (`cuda:0` when available, else `cpu`) are printed before inference.

In [ ]:
from qwen3_embedding_pipeline import MODEL_ID, MODEL_KEY, MODEL_REVISION, Qwen3EmbeddingPipeline, cosine_similarity, stage_missing_files, verify_snapshot
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'embedding_dim': EMBEDDING_DIM})
WEIGHTS_DIR = ROOT / 'weights' / MODEL_KEY
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
print({'model_id': snapshot['modelId'], 'revision': snapshot['revision'], 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes')})
pipe = Qwen3EmbeddingPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': pipe.device})

## 4. Embed and read the vectors correctly

`embed(texts, kind, instruction)` returns a dict with `embeddings` — one list per input text, in input order, each of length `dim` (1024) — plus `pooling` (`last_token`), `normalized` (`True`: every vector has unit L2 norm), `kind`, the `instruction` applied (queries only, `None` for documents), `n_tokens` per text, `truncated` flags (a text that hit the 8,192-token ceiling was cut and its vector represents only the kept prefix), and the model identity. The unit of embedding is **one vector per text**; there is no per-token or per-chunk output, and a text longer than the window is not chunked for you. Missing data has no meaning here: empty strings are rejected, not embedded.

**No intrinsic metric exists** for an embedding: the vectors are representations, and their quality can only be judged through a downstream task with labels — for retrieval, a query set with relevance judgements (nDCG, recall@k); for classification or clustering, labelled texts. The repository ships the `cosine_similarity` helper (not a metric) and no labelled data. Below, each query is compared with both documents by cosine as a **qualitative check** that the contract works: the matching document should score higher than the unrelated one. Cosine values are similarities in `[-1, 1]` on this model's geometry, not probabilities and not calibrated; absolute values are not comparable across models, and a threshold for "relevant" is the caller's to set on labelled data. Inference is deterministic on a fixed device and dtype (no sampling, `torch.inference_mode`); float32 (CPU) and bfloat16 (CUDA) differ in the second or third decimal. As recorded in the model card, the repository's CPU smoke on these four texts produced the cosine matrix `[[0.7646, 0.1414], [0.1355, 0.6000]]`, equal to the upstream README's printed values to four decimals; that is one observation on four sentences, not a retrieval score. Look for two unit-norm vectors per kind, no truncation, and the diagonal of the matrix dominating on the default sample.

In [ ]:
query_result = pipe.embed(queries, kind='query', instruction=DEFAULT_QUERY_INSTRUCTION)
document_result = pipe.embed(documents, kind='document')
query_vectors = np.asarray(query_result['embeddings'], dtype=np.float32)
document_vectors = np.asarray(document_result['embeddings'], dtype=np.float32)
print({'query_shape': query_vectors.shape, 'document_shape': document_vectors.shape, 'dim': document_result['dim'], 'pooling': document_result['pooling'], 'normalized': document_result['normalized'], 'norms': [round(float(v), 4) for v in np.linalg.norm(np.vstack([query_vectors, document_vectors]), axis=1)], 'device': pipe.device})
print({'n_tokens': dict(zip(query_ids + document_ids, query_result['n_tokens'] + document_result['n_tokens'], strict=True)), 'truncated': dict(zip(query_ids + document_ids, query_result['truncated'] + document_result['truncated'], strict=True))})
if any(query_result['truncated'] + document_result['truncated']):
    print('NOTE: at least one text hit MAX_TEXT_TOKENS and was truncated; its vector represents the kept prefix only.')
similarity = cosine_similarity(query_result['embeddings'], document_result['embeddings'])
for query_id, row in zip(query_ids, similarity, strict=True):
    print(query_id, {document_id: round(value, 4) for document_id, value in zip(document_ids, row, strict=True)})
metrics = {}
print('No metric is computed: embeddings are representations; the cosine table above is a qualitative check, and a retrieval or classification score needs labelled data.')

## 5. Export identifiers alongside vectors, and provenance

The vectors are written as CSV (`outputs/qwen3_embedding_vectors.csv`) with one row per text — `id`, `kind`, `n_tokens`, `truncated`, then `e0000…e1023` — so every vector stays attached to its identifier for downstream use. Machine-readable JSON preserves the identified texts, the query instruction, the cosine table keyed by identifier, the per-text token counts and truncation flags, the corpus digest, the repository revision, the model identifier, the immutable model revision, and the runtime identity (Python, `torch`, `transformers`, device and the precision implied by it). No credentials are recorded.

In [ ]:
import csv
import json
os.makedirs('outputs', exist_ok=True)
with open('outputs/qwen3_embedding_vectors.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['id', 'kind', 'n_tokens', 'truncated'] + [f'e{i:04d}' for i in range(EMBEDDING_DIM)])
    for kind, ids, result in (('query', query_ids, query_result), ('document', document_ids, document_result)):
        for text_id, vector, n_tokens, truncated in zip(ids, result['embeddings'], result['n_tokens'], result['truncated'], strict=True):
            writer.writerow([text_id, kind, n_tokens, truncated] + [f'{value:.7f}' for value in vector])
payload = {
    'texts': {**dict(zip(query_ids, queries, strict=True)), **dict(zip(document_ids, documents, strict=True))},
    'query_instruction': query_result['instruction'],
    'embedding_contract': {'dim': document_result['dim'], 'pooling': document_result['pooling'], 'normalized': document_result['normalized'], 'unit': 'one vector per text'},
    'n_tokens': dict(zip(query_ids + document_ids, query_result['n_tokens'] + document_result['n_tokens'], strict=True)),
    'truncated': dict(zip(query_ids + document_ids, query_result['truncated'] + document_result['truncated'], strict=True)),
    'cosine_similarity': {query_id: dict(zip(document_ids, row, strict=True)) for query_id, row in zip(query_ids, similarity, strict=True)},
    'vectors_file': 'outputs/qwen3_embedding_vectors.csv',
    'metrics': metrics,
    'sample': {'kind': sample_kind, 'name': corpus_name, 'corpus_sha256': corpus_sha256},
    'repository_revision': REPO_SHA,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
        'precision': 'bfloat16' if pipe.device.startswith('cuda') else 'float32',
    },
}
with open('outputs/qwen3_embedding_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The vectors are representations of the texts in this model's 1024-dimensional space: they predict nothing, carry no labels, and their only meaning is relative — cosine between two vectors from the same model and the same instruction. The cosine table on the default sample shows that the contract works on four short English sentences; it is not a retrieval score, and it must not be generalised to other languages, domains, long documents (truncated at 8,192 tokens), or a different query instruction, which changes the query vectors. Values are uncalibrated similarities, absolute levels are model-specific, and any relevance threshold belongs to the caller and to labelled data. Vectors from the CPU (float32) and CUDA (bfloat16) paths are close but not bitwise equal. The pipeline provides no reranking, generation, classification, chunking, dimension truncation, or training capability.

Successful execution proves that the recorded repository revision can acquire the pinned model, validate the demonstrated input, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** change `DEFAULT_QUERY_INSTRUCTION` to a task-specific instruction (for example a code-search task) in the `embed` call and watch the cosine table move; enable `USE_BYOD` with a small corpus file and your own query, then hand-label which lines are relevant to compute recall@k yourself — the first step towards a real retrieval number; embed the same document as `kind='query'` and as `kind='document'` to see how much the instruction prefix shifts a vector.

## References

- Repository README: `../README.md`
- Repository model card: `../MODEL_CARD.md`
- Weight provenance: `../docs/WEIGHTS.md`
- Upstream model: https://huggingface.co/Qwen/Qwen3-Embedding-0.6B
- Upstream code: https://github.com/QwenLM/Qwen3-Embedding
- Qwen3 Embedding: Advancing Text Embedding and Reranking Through Foundation Models (2025): https://arxiv.org/abs/2506.05176